# 06 — Comparação de Modelos
### GalaxyNet — Classificador Morfológico de Galáxias

Análise comparativa dos três modelos treinados:
- **MLP** — features tabulares fotométricas (15 features)
- **CNN** — imagens $(64 \times 64 \times 3)$ nas bandas $g$, $r$, $i$
- **Hybrid (CNN + MLP)** — fusão tardia de ambas as modalidades

Compara accuracy, completeness (recall), reliability (precision) por classe
e visualiza os padrões de erro de cada abordagem.

In [ ]:
import sys
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

def _find_src_dir():
    current = os.path.abspath(os.getcwd())
    for _ in range(5):
        candidate = os.path.join(current, 'src')
        if os.path.isdir(candidate):
            return candidate
        current = os.path.dirname(current)
    raise RuntimeError("Diretório src/ não encontrado.")

src_path = _find_src_dir()
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
)

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Caminhos
_project_root = os.path.dirname(src_path)
MODELS_DIR  = os.path.join(_project_root, 'models')
REPORTS_DIR = os.path.join(_project_root, 'reports', 'figures')

print(f"TensorFlow : {tf.__version__}")
print(f"MODELS_DIR : {MODELS_DIR}")

---
## 1. Carregar Modelos e Dados de Teste

Cada notebook (03–05) salvou o modelo treinado (`.h5`) e os dados de teste (`.pkl`).
Carregamos tudo aqui sem retreinar.

In [ ]:
# LabelEncoder
with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)
class_names = list(label_encoder.classes_)
print(f"Classes: {class_names}")

# ── MLP ───────────────────────────────────────────────────────────────────
mlp_model = keras.models.load_model(os.path.join(MODELS_DIR, 'mlp_galaxy_classifier.h5'))
with open(os.path.join(MODELS_DIR, 'mlp_test_data.pkl'), 'rb') as f:
    mlp_data = pickle.load(f)
print(f"MLP  — X_test: {mlp_data['X_test'].shape}, y_test: {mlp_data['y_test'].shape}")

# ── CNN ───────────────────────────────────────────────────────────────────
cnn_model = keras.models.load_model(os.path.join(MODELS_DIR, 'cnn_galaxy_classifier.h5'))
with open(os.path.join(MODELS_DIR, 'cnn_test_data.pkl'), 'rb') as f:
    cnn_data = pickle.load(f)
print(f"CNN  — X_test: {cnn_data['X_test'].shape}, y_test: {cnn_data['y_test'].shape}")

# ── Hybrid ────────────────────────────────────────────────────────────────
hybrid_model = keras.models.load_model(os.path.join(MODELS_DIR, 'hybrid_galaxy_classifier.h5'))
with open(os.path.join(MODELS_DIR, 'hybrid_test_data.pkl'), 'rb') as f:
    hybrid_data = pickle.load(f)
print(f"Hybrid — X_tab: {hybrid_data['X_tab_test'].shape}, "
      f"X_img: {hybrid_data['X_img_test'].shape}, y_test: {hybrid_data['y_test'].shape}")

---
## 2. Predições e Métricas por Modelo

Gera predições e calcula métricas unificadas para os três modelos.

In [ ]:
def compute_metrics(model, X_test, y_test_oh, label_encoder, model_name):
    """Computa predições e métricas para um modelo."""
    y_pred_oh = model.predict(X_test, verbose=0)
    y_pred = label_encoder.inverse_transform(np.argmax(y_pred_oh, axis=1))
    y_true = label_encoder.inverse_transform(np.argmax(y_test_oh, axis=1))

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=label_encoder.classes_)

    # Completeness (Recall) e Reliability (Precision) por classe
    per_class = []
    for i, cls in enumerate(label_encoder.classes_):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        completeness = tp / (tp + fn) if (tp + fn) > 0 else 0
        reliability  = tp / (tp + fp) if (tp + fp) > 0 else 0
        per_class.append({'Class': cls, 'Completeness': completeness,
                          'Reliability': reliability, 'N_True': cm[i, :].sum()})

    return {
        'name': model_name,
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm,
        'y_pred': y_pred,
        'y_true': y_true,
        'per_class': per_class,
    }

# Computar métricas
results = {}
results['MLP'] = compute_metrics(
    mlp_model, mlp_data['X_test'], mlp_data['y_test'], label_encoder, 'MLP')
results['CNN'] = compute_metrics(
    cnn_model, cnn_data['X_test'], cnn_data['y_test'], label_encoder, 'CNN')
results['Hybrid'] = compute_metrics(
    hybrid_model,
    [hybrid_data['X_tab_test'], hybrid_data['X_img_test']],
    hybrid_data['y_test'], label_encoder, 'Hybrid')

for name, r in results.items():
    print(f"{name:8s} — Accuracy: {r['accuracy']:.4f}, "
          f"F1 Macro: {r['f1_macro']:.4f}, F1 Weighted: {r['f1_weighted']:.4f}")

---
## 3. Tabela Comparativa de Métricas

In [ ]:
# Tabela geral
summary_rows = []
for name, r in results.items():
    row = {
        'Model': name,
        'Accuracy': f"{r['accuracy']:.3f}",
        'F1 Macro': f"{r['f1_macro']:.3f}",
        'F1 Weighted': f"{r['f1_weighted']:.3f}",
    }
    for pc in r['per_class']:
        row[f"Compl. {pc['Class']}"] = f"{pc['Completeness']:.3f}"
        row[f"Reliab. {pc['Class']}"] = f"{pc['Reliability']:.3f}"
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("TABELA COMPARATIVA DE MÉTRICAS")
print("=" * 80)
print(df_summary.to_string(index=False))

# Salvar como CSV
csv_path = os.path.join(REPORTS_DIR, 'fig22_model_comparison_table.csv')
df_summary.to_csv(csv_path, index=False)
print(f"\nTabela salva em: {csv_path}")

---
## 4. Completeness e Reliability por Classe e Modelo

Visualização das métricas científicas agrupadas por classe morfológica.
Mostra se o modelo híbrido consegue recuperar melhor as classes minoritárias
(Spiral e Irregular) em comparação com MLP e CNN isolados.

In [ ]:
# Preparar DataFrame para barplot
bar_rows = []
for name, r in results.items():
    for pc in r['per_class']:
        bar_rows.append({
            'Model': name,
            'Class': pc['Class'],
            'Completeness (Recall)': pc['Completeness'],
            'Reliability (Precision)': pc['Reliability'],
        })

df_bar = pd.DataFrame(bar_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Completeness
sns.barplot(data=df_bar, x='Class', y='Completeness (Recall)', hue='Model',
            ax=axes[0], palette='Set2')
axes[0].set_title('Completeness (Recall) por Classe', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].legend(title='Model')
axes[0].grid(True, alpha=0.3)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.2f', fontsize=9)

# Reliability
sns.barplot(data=df_bar, x='Class', y='Reliability (Precision)', hue='Model',
            ax=axes[1], palette='Set2')
axes[1].set_title('Reliability (Precision) por Classe', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].legend(title='Model')
axes[1].grid(True, alpha=0.3)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.2f', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig23_completeness_reliability_comparison.png'),
            bbox_inches='tight')
plt.show()
print('Figura salva: fig23_completeness_reliability_comparison.png')

---
## 5. Confusion Matrices — Comparação Side-by-Side

As três matrizes de confusão na mesma escala de cor permitem identificar
padrões de erro específicos de cada arquitetura.

In [ ]:
# Encontrar vmax comum para mesma escala de cor
vmax = max(r['confusion_matrix'].max() for r in results.values())

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (name, r) in zip(axes, results.items()):
    cm = r['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, vmin=0, vmax=vmax,
                annot_kws={'size': 14, 'fontweight': 'bold'})
    ax.set_title(f'{name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

fig.suptitle('Confusion Matrices — MLP vs CNN vs Hybrid',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig24_confusion_matrices_comparison.png'),
            bbox_inches='tight')
plt.show()
print('Figura salva: fig24_confusion_matrices_comparison.png')

---
## 6. Accuracy e F1-Score — Resumo Visual

In [ ]:
model_names = list(results.keys())
accuracies  = [results[m]['accuracy'] for m in model_names]
f1_macros   = [results[m]['f1_macro'] for m in model_names]
f1_weighted = [results[m]['f1_weighted'] for m in model_names]

x = np.arange(len(model_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width, accuracies, width, label='Accuracy', color='#66c2a5')
bars2 = ax.bar(x, f1_macros, width, label='F1 Macro', color='#fc8d62')
bars3 = ax.bar(x + width, f1_weighted, width, label='F1 Weighted', color='#8da0cb')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Desempenho Geral — MLP vs CNN vs Hybrid',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=12)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2, bars3]:
    ax.bar_label(bars, fmt='%.3f', fontsize=10, padding=3)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig25_accuracy_f1_comparison.png'),
            bbox_inches='tight')
plt.show()
print('Figura salva: fig25_accuracy_f1_comparison.png')

---
## 7. Classification Reports Detalhados

In [ ]:
for name, r in results.items():
    print(f"\n{'=' * 55}")
    print(f"Classification Report — {name}")
    print('=' * 55)
    print(classification_report(r['y_true'], r['y_pred'],
                                target_names=class_names, zero_division=0))

---
## 8. Resumo e Conclusões

In [ ]:
print("=" * 65)
print("RESUMO — COMPARAÇÃO DE MODELOS")
print("=" * 65)
print()

# Melhor modelo por métrica
best_acc = max(results, key=lambda m: results[m]['accuracy'])
best_f1  = max(results, key=lambda m: results[m]['f1_macro'])

print(f"Melhor Accuracy    : {best_acc} ({results[best_acc]['accuracy']:.4f})")
print(f"Melhor F1 Macro    : {best_f1} ({results[best_f1]['f1_macro']:.4f})")
print()

# Tabela resumo
print(f"{'Model':<10} {'Accuracy':>10} {'F1 Macro':>10} {'F1 Weighted':>12}")
print("-" * 45)
for name, r in results.items():
    print(f"{name:<10} {r['accuracy']:>10.4f} {r['f1_macro']:>10.4f} {r['f1_weighted']:>12.4f}")

print()
print("Métricas por classe (Completeness / Reliability):")
print(f"{'Model':<10}", end="")
for cls in class_names:
    print(f"  {cls:>12}", end="")
print()
print("-" * (10 + 14 * len(class_names)))
for name, r in results.items():
    print(f"{name:<10}", end="")
    for pc in r['per_class']:
        print(f"  {pc['Completeness']:.2f}/{pc['Reliability']:.2f}", end="")
    print()

print()
print("Artefatos salvos:")
print("  1. reports/figures/fig22_model_comparison_table.csv")
print("  2. reports/figures/fig23_completeness_reliability_comparison.png")
print("  3. reports/figures/fig24_confusion_matrices_comparison.png")
print("  4. reports/figures/fig25_accuracy_f1_comparison.png")

---
## Discussão

**Observações esperadas:**

1. **Classe majoritária (Elliptical)**: todos os modelos devem ter alta completeness/reliability,
   dado o forte desbalanceamento (86% do dataset).

2. **Classes minoritárias (Spiral, Irregular)**: o desempenho varia significativamente.
   O class weighting ajuda, mas com apenas 17 Spirals e 4 Irregulars no dataset total,
   a avaliação nestas classes é inerentemente ruidosa.

3. **MLP vs CNN**: o MLP usa features fotométricas globais (magnitudes, cores, concentração)
   enquanto a CNN aprende diretamente das imagens. A CNN pode capturar informação
   morfológica que features escalares não representam (braços espirais, assimetria).

4. **Hybrid**: a fusão tardia permite combinar o melhor de ambas as modalidades.
   Espera-se que o modelo híbrido tenha desempenho igual ou superior aos modelos
   individuais, especialmente nas classes minoritárias.

5. **Limitações**: o dataset pequeno (156 galáxias) limita a generalização.
   Resultados devem ser interpretados com cautela, especialmente para Irregular (N=4).